In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/goodreads_data.csv")
df.head()

,Unnamed: 0,Book,Author,Description,Genres,Avg_Rating,Num_Ratings,URL
0,0,To Kill a Mockingbird,Harper Lee,The unforgettable novel of a childhood in a sl...,"['Classics', 'Fiction', 'Historical Fiction', ...",4.27,"5,691,311",https://www.goodreads.com/book/show/2657.To_Ki...
1,1,Harry Potter and the Philosopher’s Stone (Harr...,J.K. Rowling,Harry Potter thinks he is an ordinary boy - un...,"['Fantasy', 'Fiction', 'Young Adult', 'Magic',...",4.47,"9,278,135",https://www.goodreads.com/book/show/72193.Harr...
2,2,Pride and Prejudice,Jane Austen,"Since its immediate success in 1813, Pride and...","['Classics', 'Fiction', 'Romance', 'Historical...",4.28,"3,944,155",https://www.goodreads.com/book/show/1885.Pride...
3,3,The Diary of a Young Girl,Anne Frank,Discovered in the attic in which she spent the...,"['Classics', 'Nonfiction', 'History', 'Biograp...",4.18,"3,488,438",https://www.goodreads.com/book/show/48855.The_...
4,4,Animal Farm,George Orwell,Librarian's note: There is an Alternate Cover ...,"['Classics', 'Fiction', 'Dystopia', 'Fantasy',...",3.98,"3,575,172",https://www.goodreads.com/book/show/170448.Ani...


In [3]:
df.columns

Index(['Unnamed: 0', 'Book', 'Author', 'Description', 'Genres', 'Avg_Rating',
       'Num_Ratings', 'URL'],
      dtype='object')

In [4]:
df = df.drop(columns=['Unnamed: 0', 'URL'])
df = df.rename(columns={'Avg_Rating':'avgr',
                        'Num_Ratings':'numr'})


In [5]:
df.isnull().sum()


Book            0
Author          0
Description    77
Genres          0
avgr            0
numr            0
dtype: int64

In [6]:
df = df.dropna()
df.isnull().sum()

Book           0
Author         0
Description    0
Genres         0
avgr           0
numr           0
dtype: int64

In [7]:
df = df[['Book', 'Author', 'Description', 'Genres', 'avgr', 'numr']]


In [8]:

df = df[df["avgr"] > 3.5]

In [9]:
## DATA CLEANING 1
# lowercase, punctuation remove, [] remove 

import re
import string


df["Genres"] = df["Genres"].apply(lambda x: re.sub(r"[\[\]']", "", str(x)))

df["Description"] = df["Description"].apply(lambda x: " ".join(x.split()[:150]))

df["Genres_list"] = df["Genres"].apply(lambda x: [g.strip() for g in x.split(",")])
## remove [] eg. a[b]c -- abc
IGNORE_GENRES = {
    "Fiction", "Childrens", "Middle Grade", "Novels", "Classics", "Historical", "Historical Fiction"
}
def get_specific_genres(genres_list):
    filtered = [g for g in genres_list if g not in IGNORE_GENRES]
    return " ".join(filtered) if filtered else " ".join(genres_list)

df["specific_genres"] = df["Genres_list"].apply(get_specific_genres)

df["content"] = df["Author"] + " " + df["specific_genres"]*3 + " " + df["Description"]
df["content"] = df["content"].str.lower()
df["content"] = df["content"].str.translate(str.maketrans('','', string.punctuation))

df["numr"] = df["numr"].str.replace(",", "").astype(int)
# maketrans(x,y,z) 
#x characters to replace 
#y	replacement characters 
#z characters to delete
df.head()


,Book,Author,Description,Genres,avgr,numr,Genres_list,specific_genres,content
0,To Kill a Mockingbird,Harper Lee,The unforgettable novel of a childhood in a sl...,"Classics, Fiction, Historical Fiction, School,...",4.27,5691311,"[Classics, Fiction, Historical Fiction, School...",School Literature Young Adult,harper lee school literature young adultschool...
1,Harry Potter and the Philosopher’s Stone (Harr...,J.K. Rowling,Harry Potter thinks he is an ordinary boy - un...,"Fantasy, Fiction, Young Adult, Magic, Children...",4.47,9278135,"[Fantasy, Fiction, Young Adult, Magic, Childre...",Fantasy Young Adult Magic,jk rowling fantasy young adult magicfantasy yo...
2,Pride and Prejudice,Jane Austen,"Since its immediate success in 1813, Pride and...","Classics, Fiction, Romance, Historical Fiction...",4.28,3944155,"[Classics, Fiction, Romance, Historical Fictio...",Romance Literature Audiobook,jane austen romance literature audiobookromanc...
3,The Diary of a Young Girl,Anne Frank,Discovered in the attic in which she spent the...,"Classics, Nonfiction, History, Biography, Memo...",4.18,3488438,"[Classics, Nonfiction, History, Biography, Mem...",Nonfiction History Biography Memoir Holocaust,anne frank nonfiction history biography memoir...
4,Animal Farm,George Orwell,Librarian's note: There is an Alternate Cover ...,"Classics, Fiction, Dystopia, Fantasy, Politics...",3.98,3575172,"[Classics, Fiction, Dystopia, Fantasy, Politic...",Dystopia Fantasy Politics School Literature,george orwell dystopia fantasy politics school...


In [10]:
##REMOVE STOPWORD
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df["content"] = df["content"].apply(lambda x: " ".join(word for word in x.split() if word not in stop_words))
#" ".join(words) -- combines word to sentence with space
# split()-- each word in x list(convert sentence into list)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
import numpy as np

df["popular"] = np.log1p(df["numr"])
df = df.sort_values('avgr', ascending=False)
df = df.drop_duplicates(subset='Book', keep='first')
df = df.reset_index(drop=True)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)
tfidfnum = tfidf.fit_transform(df['content'])  # already sparse
tfidfnum = sp.csr_matrix(tfidfnum) 
specific_genres_list = df["Genres_list"].apply(
    lambda x: [g for g in x if g not in IGNORE_GENRES] or x
)

In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(specific_genres_list)
genre_matrix = sp.csr_matrix(genre_matrix)     # convert to sparse
assert len(df) == tfidfnum.shape[0] == genre_matrix.shape[0], "ROW MISMATCH!"
print(f"TFIDF matrix shape:  {tfidfnum.shape}")
print(f"Genre matrix shape:  {genre_matrix.shape}")

TFIDF matrix shape:  (9461, 5000)
Genre matrix shape:  (9461, 615)


In [14]:
import joblib
joblib.dump(tfidfnum, "model/tfidf_matrix.pkl")
joblib.dump(genre_matrix, "model/genre_matrix.pkl")
joblib.dump(df, "model/books.pkl")
print("saved")


saved


In [17]:
IGNORE_GENRES = {
     "Childrens", "Middle Grade",
    "Audiobook", "Nonfiction", "Adult","Contemporary"
}

IMPORTANT_GENRES = {
    "Fantasy", "Magic", "Adventure","Historical Fiction" 
    "Science Fiction", "Horror", "Mystery", "Romance",
    "Thriller","Classics","War","Fiction","Historical"
}

In [18]:
def rec(bookTitle):

    matches = df[df['Book'].str.contains(bookTitle, case=False, na=False)]

    if matches.empty:
        return ["Book not found"]

    pos          = df.index.get_loc(matches.index[0])
    genres       = set(matches.iloc[0]["Genres_list"])
    title        = matches.iloc[0]["Book"]
    query_author = matches.iloc[0]["Author"]
    series       = ""

    if "(" in title:
        series = title.split("(")[1].split(",")[0].strip().lower()

    tfidf_sim = cosine_similarity(tfidfnum[pos], tfidfnum).flatten()
    genre_sim  = cosine_similarity(genre_matrix[pos], genre_matrix).flatten()
    combined   = 0.7 * tfidf_sim + 0.3 * genre_sim

    sim = sorted(list(enumerate(combined)), key=lambda x: x[1], reverse=True)

    genres_filtered = {g for g in genres if g not in IGNORE_GENRES}
    if not genres_filtered:
        genres_filtered = genres

    recs        = []
    seen_titles = set()
    seen_series = set()

    for i in sim[1:]:

        bookIndex  = i[0]
        score      = i[1]
        name       = df.iloc[bookIndex]['Book']
        name_lower = name.lower()
        author     = df.iloc[bookIndex]['Author']

        
        if bookTitle.lower() in name_lower:
            continue

        if series and series in name_lower:
            continue

       
        if name in seen_titles:
            continue

        # Skip if already have a book from this series
        candidate_series = ""
        if "(" in name:
            candidate_series = name.split("(")[1].split(",")[0].strip().lower()
        if candidate_series and candidate_series in seen_series:
            continue

        rating     = df.iloc[bookIndex]['avgr']
        popular    = df.iloc[bookIndex]['popular']
        num_rating = df.iloc[bookIndex]['numr']

        genre          = set(df.iloc[bookIndex]['Genres_list'])
        genre_filtered = {g for g in genre if g not in IGNORE_GENRES}

        if not genre_filtered:
            continue

        comGenre = genres_filtered.intersection(genre_filtered)

        if not comGenre:
            continue

        wt = len(comGenre)
        for g in comGenre:
            if g in IMPORTANT_GENRES:
                wt += 2

        final = score * (rating / 5) * (1 + wt / 5) * (1 + popular / 10)

        recs.append({
            "title":  name,
            "author": author,
            "rating": rating,
            "votes":  num_rating,
            "genres": ", ".join(comGenre),
            "score":  round(final, 4)
        })

        seen_titles.add(name)
        if candidate_series:
            seen_series.add(candidate_series)

        if len(recs) == 50:
            break

    recs = sorted(recs, key=lambda x: (x["score"] * 0.6 + x["rating"] / 5 * 0.4), reverse=True)[:5]

    if not recs:
        return ["No recommendations found for this book."]

    return recs

In [19]:
rec('harry potter')

[{'title': 'Fantastic Beasts and Where to Find Them: The Original Screenplay (Fantastic Beasts: The Original Screenplay, #1)',
  'author': 'J.K. Rowling',
  'rating': np.float64(4.18),
  'votes': np.int64(153274),
  'genres': 'Young Adult, Adventure, Fiction, Magic, Fantasy',
  'score': np.float64(3.0855)},
 {'title': 'The Magic Faraway Tree (The Faraway Tree, #2)',
  'author': 'Enid Blyton',
  'rating': np.float64(4.29),
  'votes': np.int64(36040),
  'genres': 'Young Adult, Adventure, Fiction, Magic, Fantasy',
  'score': np.float64(2.9946)},
 {'title': 'The Blue Sword (Damar, #1)',
  'author': 'Robin McKinley',
  'rating': np.float64(4.21),
  'votes': np.int64(59778),
  'genres': 'Young Adult, Adventure, Fiction, Magic, Fantasy',
  'score': np.float64(2.8101)},
 {'title': 'Alanna: The First Adventure (Song of the Lioness, #1)',
  'author': 'Tamora Pierce',
  'rating': np.float64(4.27),
  'votes': np.int64(126593),
  'genres': 'Young Adult, Adventure, Fiction, Magic, Fantasy',
  'score

Jane Eyre found at rank: 10
Score: 0.3775
Genres: ['Classics', 'Fiction', 'Romance', 'Historical Fiction', 'Gothic', 'Literature', 'Historical']
